# California Housing Price Prediction ML Project
### Synent Technologies - Data Science Internship (Summer 2026)
### Task 8: Machine Learning Model (House Price Prediction)

This notebook demonstrates the complete end-to-end Machine Learning pipeline to predict house prices in California using the California Housing Dataset (`housing.csv`).

**Objective:** Build, evaluate, and compare multiple regression models (Linear Regression, Decision Tree, and Random Forest) to forecast median house values, select the champion model, and package it for inference.

### 1. Environment Setup & Library Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

%matplotlib inline
sns.set_theme(style="whitegrid")

### 2. Dataset Loading
We load the raw California Housing dataset from `data/raw/housing.csv`.

In [ ]:
file_path = "../data/raw/housing.csv"
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f'Dataset successfully loaded. Shape: {df.shape}')
else:
    print(f'Error: Dataset not found at {file_path}')
df.head()

### 3. Data Preprocessing & Cleaning
- Check and impute missing numerical values (the `total_bedrooms` column has 207 missing values).
- Filter out any physically impossible values (e.g. non-positive coordinates, population, etc.).
- Filter extreme physical outliers (e.g., cases where `total_bedrooms` exceeds `total_rooms`).

In [ ]:
# Inspect missing values
print("Missing values:")
print(df.isnull().sum())

# Impute missing total_bedrooms with its median value
median_bedrooms = df["total_bedrooms"].median()
print(f'Imputing total_bedrooms with training median: {median_bedrooms}')
df["total_bedrooms"] = df["total_bedrooms"].fillna(median_bedrooms)

# Basic integrity checks: ensure numeric counts are strictly positive
non_negative_cols = [
    "housing_median_age", "total_rooms", "total_bedrooms", 
    "population", "households", "median_income", "median_house_value"
]
initial_len = len(df)
for col in non_negative_cols:
    df = df[df[col] > 0]
print(f'Removed {initial_len - len(df)} rows with negative or zero values.')

# Remove records where total_bedrooms is greater than total_rooms
initial_len = len(df)
df = df[df["total_bedrooms"] <= df["total_rooms"]]
print(f'Removed {initial_len - len(df)} rows where total_bedrooms was greater than total_rooms.')
print(f'Cleaned dataset shape: {df.shape}')

### 4. Feature Engineering & Categorical Encoding
- We derive ratio features (`rooms_per_household`, `bedrooms_per_room`, and `population_per_household`) to capture neighborhood densities and dwelling scales.
- We use scikit-learn's `OneHotEncoder` to encode the categorical `ocean_proximity` column, ensuring stable encoding parameters for test and production environments.

In [ ]:
# Derive ratio features
epsilon = 1e-5
df["rooms_per_household"] = df["total_rooms"] / (df["households"] + epsilon)
df["bedrooms_per_room"] = df["total_bedrooms"] / (df["total_rooms"] + epsilon)
df["population_per_household"] = df["population"] / (df["households"] + epsilon)
print("Created derived columns: ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']")

# Categorical One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_arr = encoder.fit_transform(df[["ocean_proximity"]])
encoded_cols = encoder.get_feature_names_out(["ocean_proximity"])
encoded_df = pd.DataFrame(encoded_arr, columns=encoded_cols, index=df.index)

# Drop original categorical column and concatenate
df_processed = df.drop(columns=["ocean_proximity"])
df_processed = pd.concat([df_processed, encoded_df], axis=1)
print(f'Processed columns: {df_processed.columns.tolist()}')
df_processed.head()

### 5. Dataset Splitting
We split the preprocessed dataset into features (`X`) and target variable (`y` = `median_house_value`), followed by an 80/20 train/test split.

In [ ]:
X = df_processed.drop(columns=["median_house_value"])
y = df_processed["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features')

### 6. Model Training
We train three regressors: Linear Regression, a Decision Tree, and a Random Forest (ensemble).

In [ ]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
print("Linear Regression training complete.")

# 2. Decision Tree Regressor (capped max_depth to prevent extreme overfitting)
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
print("Decision Tree training complete.")

# 3. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Random Forest training complete.")

### 7. Evaluation & Comparison
We score the test splits and compare Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and R-squared coefficient ($R^2$).

In [ ]:
models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf
}

metrics = {}
predictions = {}

for name, model in models.items():
    preds = model.predict(X_test)
    predictions[name] = preds
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    metrics[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    print(f'{name:20} -> MAE: ${mae:11.2f} | RMSE: ${rmse:11.2f} | R2: {r2:.4f}')

# Convert to comparative DataFrame
comp_df = pd.DataFrame.from_dict(metrics, orient="index").reset_index().rename(columns={"index": "Model"})

# Plot comparisons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(ax=axes[0], x="Model", y="RMSE", data=comp_df, palette="Blues_r")
axes[0].set_title("RMSE Comparison (Lower is Better)")
axes[0].set_ylabel("RMSE in USD")

sns.barplot(ax=axes[1], x="Model", y="R2", data=comp_df, palette="Greens_r")
axes[1].set_title("R-squared Comparison (Higher is Better)")
axes[1].set_ylabel("R-squared Score")
plt.tight_layout()
plt.show()

We also plot actual vs predicted prices (residuals plot) and feature importances for our best-performing model (Random Forest).

In [ ]:
best_model_name = comp_df.loc[comp_df["R2"].idxmax(), "Model"]
best_model = models[best_model_name]
print(f'Champion Model selected: {best_model_name}')

# Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions[best_model_name], alpha=0.3, color="#4A90E2", edgecolors="w", s=30)
min_val = min(y_test.min(), predictions[best_model_name].min())
max_val = max(y_test.max(), predictions[best_model_name].max())
plt.plot([min_val, max_val], [min_val, max_val], color="#D0021B", linestyle="--", linewidth=2, label="Perfect Fit Line")
plt.title(f"Actual vs. Predicted Prices ({best_model_name})")
plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot
if hasattr(best_model, "feature_importances_"):
    feat_importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
    feat_importances = feat_importances.sort_values(ascending=True)
    plt.figure(figsize=(10, 6))
    feat_importances.plot(kind="barh", color="#50E3C2", edgecolor="grey")
    plt.title(f"Feature Importances - {best_model_name}")
    plt.xlabel("Relative Importance")
    plt.tight_layout()
    plt.show()

### 8. Exporting the Model
We save the champion model and preprocessor metadata objects using `joblib` so that they can be integrated into prediction pipelines or web applications.

In [ ]:
os.makedirs("../models", exist_ok=True)
model_path = "../models/house_price_model.joblib"
preprocessor_path = "../models/preprocessor.joblib"

joblib.dump(best_model, model_path)
preprocessor_metadata = {
    "imputer_values": {"total_bedrooms": median_bedrooms},
    "categorical_cols": ["ocean_proximity"],
    "encoder": encoder,
    "feature_names_out": df_processed.columns.tolist()
}
joblib.dump(preprocessor_metadata, preprocessor_path)
print("Best model and preprocessor objects serialized successfully!")